In [ ]:
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt
!pip install Sastrawi
!pip install emoji
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords as stopwords_scratch
from sklearn.feature_extraction.text import TfidfVectorizer
import emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 43.9 MB/s eta 0:00:00


# **Stopwords**

In [ ]:
import nltk
nltk.download('stopwords')
list_stopwords = stopwords_scratch.words('indonesian')
list_stopwords_en = stopwords_scratch.words('english')
list_stopwords.extend(list_stopwords_en)
list_stopwords.extend(['ya', 'yg', 'ga', 'aja', 'yuk', 'dah', 'ngga', 'engga', 'ygy', 'gak', 'nya', 'baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya', 'bang', 'kak', 'min', 'admin', 'guys', 'guysss', 'bikin', 'udah', 'udahh', 'udh', 'bgt', 'banget', 'dong', 'sih', 'kok', 'kan', 'deh', 'lho', 'loh', 'mas', 'mbak', 'pak', 'bu', 'coy', 'bro', 'sis', 'wkwk', 'haha', 'hehe', 'hihi', 'hmm', 'trs', 'trsnya', 'tp', 'tapi', 'jd', 'jadi', 'jg', 'juga', 'aja', 'ajaa', 'ajaahh', 'ajalah', 'kalo', 'kalau', 'karna', 'karena', 'kyk', 'kayak', 'kek', 'bgt', 'btw', 'asli', 'wajib', 'yoi', 'lah', 'loh', 'dong', 'ko', 'sih', 'aja', 'yah', 'nih', 'tuh', 'emang', 'gitu', 'apalah', 'gituloh'])
stopwords_df = pd.DataFrame(list_stopwords, columns=['stopword'])
stopwords_df.to_csv('./data/stopwords.csv', index=False, header=False)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


# **Import Dataset**

In [ ]:
comment_df = pd.read_csv("./data/dataset_comment_yt.csv")
# Load stopwords and convert to a set for efficient lookup
stopword_list = pd.read_csv("./data/stopwords.csv", header=None)[0].tolist()
stopwords_set = set(stopword_list)

# **Stemmer Bahasa Indonesia**

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# **Preprocessing Data**

In [ ]:
# Fungsi pre-processing
def preprocess_text(text):
    # Pastikan input adalah string
    text = str(text)

    # Hapus emoji
    text = emoji.replace_emoji(text, replace="")

    # Konversi ke lowercase
    text = text.lower()

    # Hapus URL
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Hapus angka dan karakter khusus (selain huruf dan spasi)
    # Ini akan menghapus tanda baca, angka, dan simbol lainnya
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Normalisasi spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords (menggunakan set yang sudah dibuat)
    tokens = [word for word in tokens if word not in stopwords_set]

    # Remove very short words (kurang dari 2 karakter)
    tokens = [word for word in tokens if len(word) > 2]

    # Stemming
    tokens = [stemmer.stem(word) for word in tokens]

    # Gabungkan kembali token menjadi string
    return ' '.join(tokens)

# **Menyimpan Dalam Dataframe Dataset Yang Sudah Di Cleaning**

In [ ]:
import nltk
nltk.download('punkt_tab')

comment_df['comments'] = comment_df['comments'].astype(str)
comment_df['cleaned_comments'] = comment_df['comments'].apply(preprocess_text)

comment_df = comment_df[comment_df['cleaned_comments'].str.strip() != '']

# Tampilkan hasil pre-processing pada beberapa baris pertama
print(comment_df[['comments', 'cleaned_comments']].head())

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


                                            comments  \
0  Makin lama makin ke sini hukum semakin ga jela...   
1        Gila bnr padahal di pilih dari suara rakyat   
2  HOUWWWWW...AKAN. ANYAK RAKYAT YANG DI PENJARA....   
3    Seru bisa rakyat semua  turun kejal ini  mantap   
4  Kaya anak teka undang2..begituh ga berbobot bp...   

                                    cleaned_comments  
0  hukum jelasuu ampas aset tele tir hidup indonesia  
1                        gila bnr pilih suara rakyat  
2  hou anyak rakyat penjaradan siksa azab preside...  
3                     seru rakyat turun kejal mantap  
4  kaya anak teka undangbegituh bobot bpk undangy...  


# **Penghapusan Data Duplikat Dalam Dataset Cleaned**

In [ ]:
print("Jumlah data duplikat:", comment_df['cleaned_comments'].duplicated().sum())

comment_df = comment_df.drop_duplicates(subset='cleaned_comments')

Jumlah data duplikat: 1772


# **Penyimpanan Dataset Dalam CSV**

In [ ]:
comment_df.to_csv('./data/cleaned_comment_yt.csv', index=False)